In [1]:
!pip install pyarrow

# Kết nối drive


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# Tạo folder chứa file parquet

In [10]:
import os

output_dir = "/content/drive/MyDrive/amazon_reviews_processed"
os.makedirs(output_dir, exist_ok=True)

# Code xử lý

In [11]:
import os
import pyarrow as pa
import pyarrow.json as paj
import pyarrow.compute as pc
import pyarrow.parquet as pq

categories = [
    "Appliances",
    "Electronics",
    "Books",
    "Home_and_Kitchen"
]

columns = [
    "rating",
    "text",
    "parent_asin",
    "timestamp",
    "user_id",
    "helpful_vote",
    "verified_purchase"
]

output_dir = "/content/drive/MyDrive/amazon_reviews_processed"
os.makedirs(output_dir, exist_ok=True)

for category in categories:

    input_file = f"/content/drive/MyDrive/amazon_reviews_2023/{category}/raw/review_categories/{category}.jsonl"
    output_file = f"{output_dir}/{category}.parquet"

    # check file tồn tại
    if not os.path.exists(input_file):
        print(f"Skip {category} (file not found)")
        continue

    print("Processing:", category)

    table = paj.read_json(
        input_file,
        read_options=paj.ReadOptions(block_size=1<<26)
    )

    writer = None

    for batch in table.to_batches():

        table_batch = pa.Table.from_batches([batch])

        # 1 lọc cột
        table_batch = table_batch.select(columns)

        # 2 bỏ null
        table_batch = table_batch.filter(pc.is_valid(table_batch["text"]))
        table_batch = table_batch.filter(pc.is_valid(table_batch["rating"]))

        # 3 clean html
        text = pc.replace_substring(table_batch["text"], "<br />", " ")
        text = pc.utf8_trim_whitespace(text)

        table_batch = table_batch.set_column(
            table_batch.schema.get_field_index("text"),
            "text",
            text
        )

        # 4 helpful_vote null -> 0
        hv = pc.fill_null(table_batch["helpful_vote"], 0)

        table_batch = table_batch.set_column(
            table_batch.schema.get_field_index("helpful_vote"),
            "helpful_vote",
            hv
        )

        # 5 convert timestamp
        ts = pc.cast(table_batch["timestamp"], pa.timestamp("ms"))

        table_batch = table_batch.set_column(
            table_batch.schema.get_field_index("timestamp"),
            "timestamp",
            ts
        )

        if writer is None:
            writer = pq.ParquetWriter(
                output_file,
                table_batch.schema,
                compression="snappy"
            )

        writer.write_table(table_batch)

    if writer:
        writer.close()

    print("Finished:", category)

Finished processing!
